# TP 1: Get familiar with Keras and PyTorch, Multi-Layer Perceptron (MLP) and Convolutional Neural Network (CNN)

## Objective of the following tutorial: Implement a MLP and a CNN on CIFAR-10 data

Help:
- https://pytorch.org/tutorials/beginner/basics/intro.html
- https://pytorch.org/tutorials/beginner/deep_learning_60min_blitz.html


- [PyTorch official website](https://pytorch.org/)
- [PyTorch Lightning](https://www.pytorchlightning.ai/)

## Package check

PyTorch is a deep learning framework that provides tools for: tensor computation, automatic differentiation, neural network computation, optimization, GPU acceleration, etc

In [ ]:
import scipy
import numpy as np # manipulate N-dimensional arrays
import pandas as pd # data frame
import matplotlib.pyplot as plt # data plotting
import seaborn # advanced data plotting
from sklearn import preprocessing # basic ML models
# import scipy # scientific computing library

import os

import torch

import torchvision
import torchvision.transforms as transforms

Check environment:

In [ ]:
print(f'PyTorch version: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')

Define device:

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

# CIFAR-10 dataset

## a) Load the data

The data can be directly downloaded with PyTorch. Please visit the following website: https://pytorch.org/vision/stable/datasets.html

In [ ]:
# # With data images, the preprocess must be declared before loading the data
# For the moment, we just convert to Tensor type

transform = transforms.Compose([
    transforms.ToTensor(),
])

train_dataset = torchvision.datasets.CIFAR10(
    root="./data",
    train=True,
    download=True,
    transform=transform
)

test_dataset = torchvision.datasets.CIFAR10(
    root="./data",
    train=False,
    download=True,
    transform=transform
)

If the download of the data takes too much time, download the data manually [here](https://www.google.com/url?sa=i&source=web&rct=j&url=https://data.brainchip.com/dataset-mirror/cifar10/cifar-10-python.tar.gz&ved=2ahUKEwjg-rr14oGXAxUeOfsDHbVJL7QQy_kOeggIAggACBwQAg&opi=89978449&cd&psig=AOvVaw1dVj7ZdiiBFhFiE_10HIqO&ust=1790152242765000) and place the compressed file in a data folder.

How many samples and classes in the datasets ? 

In [ ]:
# TODO

Dimension and type of the data ?

In [ ]:
# TODO

## b) Preprocess the data

Split the training set into a train and validation set.
Have a look on https://pytorch.org/docs/stable/data.html. <br>
You can easily create your own dataset.

In [ ]:
from sklearn.model_selection import train_test_split
random_state = 42 #for reproductible results

# Split by generating train and val index with train_test_split() 
# Be cautious and keep the proportions of the original dataset
train_indices, val_indices = ...

In [ ]:
# Convert to Dataset objects
trainset = torch.utils.data.Subset(dataset, train_indices)
valset = torch.utils.data.Subset(dataset, val_indices)

(Optional) Another possibility is to use the random_split but the proportions will not be kept. Note that the more samples you use, the lower the likelihood of creating an imbalance.

In [ ]:
trainset, valset = torch.utils.data.random_split(...)

Only once you have your 3 Datsets (train, val, test), you can use the dataloader that will split each dataset into batches. 

Here, with Pytorch, we can custom our dataloaders especially to optimize training time using arguments like `num_workers`, `prefetch_factor`...etc. See the documentation: https://pytorch.org/docs/stable/data.html#torch.utils.data.DataLoader

In [ ]:
batch_size = ... # You can start with 32
train_loader = torch.utils.data.DataLoader(...)
val_loader = ...
test_loader = ...

What does the DataLoaders contain ? Print the dimension of each element:

In [ ]:
# TODO

## c) Visualize the data

Vizualize a grid of images <br>
Help: https://pytorch.org/vision/stable/auto_examples/plot_visualization_utils.html#visualizing-a-grid-of-images

In [ ]:
from torchvision.utils import make_grid
for images, _ in test_loader:
    # TODO

## d) Data normalization

When loading the data in a), we define 'transform' to just convert the data into Tensor objects. 

Neural network generally benefit from inputs with well-scaled values. 
We can further normaize each RGB channel using its mean and standard deviation:

In [ ]:
# Here, the mean and std of each channel is given but in real cases these values must be computed
#normalization: y = (x-mean)/std
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(
        mean=(0.4914, 0.4822, 0.4465),
        std=(0.2470, 0.2435, 0.2616)
    )
])

Reload the data with this transformation and apply the same steps as in b).

Visualize the data as in c):

In [ ]:
# TODO

What is the difference ?

# MLP

A MLP is composed of fully connected layers. Fully connected layers expect one-dimensional vector as input, but CIFAR-10 images are 3-dimensional $(3 \times n \times n)$.

So, we first need to flatten each input images and define the input dimension:

In [ ]:
input_dim = ...

## a) Implementation

Here, we define our model as a class object with properties like layers, activation functions...etc and always a `forward` function that is called for any forward pass through the model.

In the Pytorch documentation, authors describe `nn.Module` as:

"*Base class for all neural network modules. **Your models should also subclass this class.** Modules can also contain other Modules, allowing to nest them in a tree structure.*"

Check this documentation to understand how to design your models using pytorch `nn.Module`: https://pytorch.org/docs/stable/generated/torch.nn.Module.html

We will implement a simple architecture composed of 2 layers. The layers of the model need to follow this order:

input_data $\rightarrow$ Flatten() $\rightarrow$ Linear(input_dim) $\rightarrow$ ReLU() $\rightarrow$ Linear(512, 256) $\rightarrow$ Linear(256, nb_classes)

In [ ]:
class MLP(nn.Module):
    def __init__(self):
        super().__init__()

        self.network = nn.Sequential(
            # TODO
            # Flatten layer
            # 1st Linear and ReLU
            # 2nd Linear and ReLU
            # 3rd Linear to output
        )

    def forward(self, x):
        return self.network(x)

After creating the model class, we need to declare it and push to device:

In [ ]:
model = MLP().to(device)
print(model)

Calculate the number of parameters of the model:

In [ ]:
# TODO

## b) Loss and optimizer

We consider a multi-class classification problem, so the loss function to use is the cross-entropy:

$$ \mathcal{L}_{\mathrm{CE}} = -\frac{1}{N} \sum_{i=1}^{N} \sum_{j=1}^{C} y_{i,j} \log (p_{i,j}) $$

All standard losses are implemented directly in PyTorch, same for standard optimizers. We just have to declare them.

Note: The cross-entropy loss function implemented in PyTorch expects raw logits outputs from the neural network, so the final layer of the model should not contain a Softmax activation.

In [ ]:
# TODO
# Search into Pytorch documentation to declare the cross-entropy loss and the Adam optimizer
criterion = ...
optimizer = ...

## c) Training loop

Once we have our DataLoaders, model, loss and optimizer defined, we can train our model.
A typical training loop in PyTorch generally follows:
1. Put the model in training mode
2. Iterate over batches (over DataLoader)
3. Reset the gradients
4. Compute predictions
5. Compute the loss
6. Compute gradients using backpropagation
7. Update the model parameters

Here is an example of training loop:

In [ ]:
num_epochs = 10
train_losses = []

for epoch in range(num_epochs):
    # Put the model in training mode
    model.train()

    # runnning loss to track across batches
    running_loss = 0.0

    # Iterate over batches
    for x, y in train_loader:
        x = x.to(device) # data/images
        y = y.to(device) # labels

        # Reset gradients
        optimizer.zero_grad()

        # Compute predictions
        outputs = model(x)

        # Compute loss
        loss = criterion(outputs, y)

        # Backpropagation
        loss.backward()

        # Update parameters
        optimizer.step()

        running_loss += loss.item()

    # average loss values
    epoch_loss = running_loss/len(train_loader)
    train_losses.append(epoch_loss)

    print(f'Epoch {epoch+1}/{num_epochs}:\n \t Loss: {epoch_loss}')
    

Implement this training process as a function that you can reuse for any model. 

In [ ]:
# TODO
# Training function for one epoch
def train(...):
    # ...
    return epoch_loss

## d) Evaluation 

The training loop implemented in c) permits to train the model for a number of fixed epoch. However, we only compute the loss using the training set. We also want to evaluate the model using the validation and test sets. 

We then need to turn the model on evaluation mode and deactivate the gradients.

Write a function to evaluate a model that returns the loss value and the accuracy:

In [ ]:
def evaluate(model, data_loader, device):
    # Put the model in eval mode
    model.eval()

    ...

    # Deactivate the gradients
    with torch.no_grad():
        for x, y in data_loader:
            # TODO
            # Remember that the last layer of the model is a Linear layer. You have to apply Softmax to obtain probabilities.

    return acc_value, loss_value

## e) Train the model

We now have all the elements to execute a complete training and evaluation process.

By using the functions that you implemented, train the MLP model for 50 epochs and evaluate it on the test set at the end. 

At each epoch, train the model on the training set and evaluate on the training and validation set.

Plot the loss value and accuracy evolution over the number of epoch for the train and validation sets. Save all the loss, accuracy values of each epoch in a pandas DataFrame to keep a training history.

In [ ]:
# TODO

# CNN

The MLP treats an image as a 1-dimensional vector.

However, images have a spatial structure.

For example, neighboring pixels are related to one another,
and local patterns such as edges and textures can be useful
for recognizing objects.

A CNN explicitly exploits this spatial structure through
convolutional layers. There is no need to flatten the input data with this architecture.

## a) Implementation

PyTorch documentation for CNN:
- https://pytorch.org/docs/stable/nn.functional.html
- [Conv2d](https://pytorch.org/docs/stable/generated/torch.nn.Conv2d.html)
- [MaxPool2d](https://pytorch.org/docs/stable/generated/torch.nn.MaxPool2d.html)
- [Dropout](https://pytorch.org/docs/stable/generated/torch.nn.Dropout.html)
- [Linear](https://pytorch.org/docs/stable/generated/torch.nn.Linear.html)

We will follow an architecture such as:

input $\rightarrow$ Conv2D(in=3, out=32) $\rightarrow$ ReLU $\rightarrow$ MaxPool2D(2x2) $\rightarrow$ Conv2D (in=32, out=64) $\rightarrow$ ReLU $\rightarrow$ MaxPool2D(2x2) $\rightarrow$ Flatten $\rightarrow$ Linear(128) $\rightarrow$ ReLU $\rightarrow$ Linear(nb_classes)

In [ ]:
import torch.nn as nn
import torch.nn.functional as F

class CNN(nn.Module):
    def __init__(self):
        super().__init__() # always subclass
        # the three arguments in_channels, out_channels, kernel_size must be filled, the others are optionnal and have default values
        # out_channels correspond to the number of filters
        # if heigth=width in the kernel size, just set one value instead of a tuple
        # stride is set to 1 by default
        # padding='same' pads the input so the output has the shape as the input. However, this mode doesn’t support any stride values other than 1.
        # We separate the model into 2 modules: feature_extractor and classifier
        self.features = nn.Sequential(
            # 1st conv layer
            # TODO
            
            # 2nd conv layer
            # TODO
        )

        self.classifier = nn.Sequential(
            # TODO
        )

    def forward(self, x):
        # TODO
        return x

cnn = CNN().to(device) # train on GPU if available

Find the number of parameters of the model:

In [ ]:
# TODO

### Understanding the dimension

How do we keep track of the image sizes after convolutions, poolings and padding ?

Pytorch documentation displays the equations to compute the image size.
Check the documentation of each layer!
https://pytorch.org/docs/stable/generated/torch.nn.Conv2d.html#torch.nn.Conv2d
https://pytorch.org/docs/stable/generated/torch.nn.MaxPool2d.html#torch.nn.MaxPool2d

Let's perform a safety check below using the equation provided by pytorch:

For one spatial dimension, the ouput size of a convolution is:
$$ D_{\mathrm{out}} = \dfrac{D_{\mathrm{in}}+2\mathrm{padding}-\mathrm{dilation}*(\mathrm{kernel}-1)-1}{\mathrm{stride}} $$

Pooling layers follow the same formula but with different parameters.

Implement a function that will compute the output dimension after convolution and after pooling layer:

In [ ]:
# Function to compute size after conv2D layer or maxpooling 2D layer and to compute the value of padding when using padding='same'
def conv_output_dim(input_size, kernel_size, padding=0, stride=1, dilation=1):
    # TODO
    return ...

def pool_output_dim(input_size, kernel_size, padding=0, stride=None, dilation=1):
    # TODO
    ## if stride == None: stride=kernel_size
    return ...

Use these functions to calculate the spatial dimensions after each operation:

In [ ]:
input_dims = 3, 32, 32
# TODO
print(f'Output dimensions after 1st Conv layer: {32, ..., ...}\n')

print(f'Output dimensions after 1st Pooling layer: {32, ..., ...}\n')

print(f'Output dimensions after 2nd Conv layer: {64, ..., ...}\n')

print(f'Output dimensions after 2nd Pooling layer: {64, ..., ...}\n')

This helps understand more what happen inside our model.

## b) Train the model

Train your CNN model in the same way as the MLP.

You can reuse the training and evaluation function. Do not forget to keep a training history.

In [ ]:
# TODO

Save the model -> useful later for transfer learning <br>
Help: https://pytorch.org/tutorials/beginner/saving_loading_models.html


In [ ]:
PATH = './cifar_cnn.pth'
torch.save(net.state_dict(), PATH)

# Compare MLP and CNN

Compare the two architectures that you trained:
1) Plot the training curves side by side
2) Compare the number of parameters
3) Final accuracies on test set


In [ ]:
# TODO

Questions:
1) Which model performs better ?
2) Which model has more parameters ?
3) How does the CNN exploit spatial structure ? Is it effective ?
4) What information is lost when flatten layer is applied ?

In [ ]:
# TODO

# Error analysis

We considered accuracy for the evaluation but this metric does not give precise error estimation.<br>
To see which classes are most frequently confused, we can use confusion matrix. A classification report containing several metrics will also be considered.

In [ ]:
import itertools
from sklearn.metrics import confusion_matrix, classification_report

In [ ]:
# function to print a confusion matrix given a sklearn confusion matrix object
def plot_confusion_matrix(cm, classes,
                          normalize=False,
                          title='Confusion matrix',
                          cmap=plt.cm.Blues, size=15):
    """
    This function prints and plots the confusion matrix.
    Normalization can be applied by setting `normalize=True`.
    from http://scikit-learn.org/stable/auto_examples/model_selection/plot_confusion_matrix.html
    :param cm: (numpy matrix) confusion matrix
    :param classes: [str]
    :param normalize: (bool)
    :param title: (str)
    :param cmap: (matplotlib color map)
    """
    if normalize:
        cm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]

    plt.figure(figsize=(size,size+2))
    im = plt.imshow(cm, interpolation='nearest', cmap=cmap)
    plt.title(title, size=18, family='serif')
    cb = plt.colorbar(im, fraction=0.046, pad=0.04)
    cb.ax.tick_params(labelsize=14)
    #Set the font type and size of the colorbar
    for l in cb.ax.yaxis.get_ticklabels():
        l.set_family("serif")

    tick_marks = np.arange(len(classes))
    plt.xticks(tick_marks, classes, rotation=90, fontname='serif', size=14)
    plt.yticks(tick_marks, classes, fontname='serif', size=14)

    fmt = '.1f' if normalize else 'd'
    thresh = cm.max() / 2.
    for i, j in itertools.product(range(cm.shape[0]), range(cm.shape[1])):
        plt.text(j, i, format(cm[i, j], fmt),
                 horizontalalignment="center",
                 color="white" if cm[i, j] > thresh else "black",
                 family='serif',
                 size=12)


    plt.ylabel('True label', size=18, fontname='serif')
    plt.xlabel('Predicted label', size=18, fontname='serif')
    plt.tight_layout()
    plt.show()

Compute the confusion matrix of one or both of your models. Normally, you don't have to retrain your models, you only need to use them in evaluation mode. You might want to collect the predictions and labels in two lists.

In [ ]:
# TODO

In [ ]:
cm = confusion_matrix(..., ...) # confusion matrix # TODO

In [ ]:
LABELS = {
    0 : "airplane",
    1 : "automobile",
    2 : "bird",
    3 : "cat",
    4 : "deer",
    5 : "dog",
    6 : "frog",
    7 : "horse",
    8 : "ship",
    9 : "truck"
}

In [ ]:
plot_confusion_matrix(cm, LABELS.values(),
                          normalize=True,
                          title='Confusion matrix',
                          cmap=plt.cm.Blues, size=7)

You can do the same for classification report: 

In [ ]:
# Report
print(classification_report(..., ..., target_names=LABELS.values())) #TODO

# Extension

## Early-stop

In the previous training functions, the training process was defined for a fixed number of epoch. Usually, we declare an early-stop that will track the loss and stop the training when it is not improving. This tool permits to avoids overfitting.

In [ ]:
# You can build your own callbacks like EarlyStopping for instance:
class EarlyStopper():
    def __init__(self, patience=1, min_delta=0):
        self.patience = patience
        self.min_delta = min_delta
        self.counter = 0
        self.min_validation_loss = float('inf')

    def early_stop(self, validation_loss):
        if validation_loss < self.min_validation_loss:
            self.min_validation_loss = validation_loss
            self.counter = 0
        elif validation_loss > (self.min_validation_loss + self.min_delta):
            self.counter += 1
            if self.counter >= self.patience:
                return True
        return False

## Schedulers

Check the documentation for built-in learning rate schedulers:

- https://pytorch.org/docs/stable/generated/torch.optim.lr_scheduler.LinearLR.html#torch.optim.lr_scheduler.LinearLR

- https://pytorch.org/docs/stable/generated/torch.optim.lr_scheduler.StepLR.html#torch.optim.lr_scheduler.StepLR

- https://pytorch.org/docs/stable/generated/torch.optim.lr_scheduler.OneCycleLR.html#torch.optim.lr_scheduler.OneCycleLR

## Data augmentation

For data augmentation, check again the transforms documentation: https://pytorch.org/vision/stable/transforms.html

In [ ]:
# Change transforms to add augmentation
# TODO

## Hyperparameters optimization 

For further optimization, check the Optuna package: https://optuna.readthedocs.io/en/stable/index.html

Read: https://medium.com/pytorch/using-optuna-to-optimize-pytorch-hyperparameters-990607385e36

- You can specify the metric you want to optimize (accuracy, f1_score...etc)
- You write your own objective function
- You specify the search type: grid, random or bayesian for instance.
- You specify the number of trials
- You can retrieve all the search results and best parameters

In [ ]:
import optuna

def objective(trial):
    # TODO
    # objective function to run at each trial.

    # Specify the hyperparameters to search, their type and the value range:
    lr = trial.suggest_loguniform("lr", 1e-5, 1e-1)
    ...

    # Train the model
    model.train()...

    # Must return the metric you want to maximize or minimize.
    return metric

# Create a search study
study = optuna.create_study()
# Run the search
study.optimize(objective, n_trials=..., )
# Check best params
study.best_params